<a href="https://colab.research.google.com/github/tecepeipe/ollama-colab-runner/blob/main/ollama_colab_runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Ollama Colab Runner**
# <img src='https://ollama.com/public/ollama.png' alt="Ollama"/>
When running this, ideally, select an instance with GPU:<br>
T4 (16GB/7B models) free tier, A100/L4 (40GB/27B models) paid tier<br><br>
Run each of the 3 cells, before running your prompt.<br>
If you interrupt execution, start the server again

In [ ]:
# @title Install components
!apt-get install -y pciutils lshw
!apt-get update
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama

!echo 'debconf debconf/frontend select Noninteractive' | sudo debconf-set-selections
!sudo apt-get update && sudo apt-get install -y cuda-drivers

import os
# Set LD_LIBRARY_PATH so the system NVIDIA library
os.environ.update({'LD_LIBRARY_PATH': '/usr/lib64-nvidia'})

In [ ]:
# @title Start server and API endpoint
import subprocess
import os
import time

# Ensure no old process is hanging around
!pkill ollama

env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0"
env["OLLAMA_ORIGINS"] = "*"

print("Starting Ollama server...")
process = subprocess.Popen(['ollama', 'serve'], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print("Server process launched in background.")

In [ ]:
# @title Select Model

model = "mistral:7b"  # @param ["granite3.1-moe:3b", "mistral:7b"]

print(f"Pulling {model}...")
!ollama pull {model}

In [ ]:
# @title Interacting with the model (streaming)
Prompt = "tell me a story"  # @param {"type":"string"}
Think = False # @param {"type":"boolean"}
Show_Thinking = False # @param {"type":"boolean"}

from IPython.display import display, Markdown
import ollama
import time
import httpx
import subprocess
import os

def ensure_server():
    try:
        httpx.get("http://localhost:11434/api/tags", timeout=1.0)
        return True
    except:
        print("⚠️ Server not found. Attempting to start Ollama...")
        env = os.environ.copy()
        env["OLLAMA_HOST"] = "0.0.0.0"
        env["OLLAMA_ORIGINS"] = "*"
        subprocess.Popen(['ollama', 'serve'], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return False

# 1. Robust check for Ollama server
print("🔍 Checking Ollama server status...")
server_ready = False
for i in range(10):
    if ensure_server():
        server_ready = True
        print("✅ Ollama server is active!")
        break
    else:
        if i == 0:
            print("Waiting for server to wake up...")
        time.sleep(3)

if not server_ready:
    print("❌ Error: Ollama server is NOT responding.")
    print("ACTION REQUIRED: Run cell 0oEhDdPXVamH manually.")
else:
    try:
        print(f"🚀 Sending prompt to model: {model}...")
        stream = ollama.chat(model=model, messages=[{"role": "user", "content": Prompt}], stream=True)

        full_response = ""
        for chunk in stream:
            if 'content' in chunk['message']:
                content = chunk['message']['content']
                full_response += content
                print(content, end="", flush=True)

        print("\n")
        display(Markdown(full_response))
    except Exception as e:
        print(f"\n❌ Inference Error: {e}")

In [ ]:
# @title Install Tailscale
!curl -fsSL https://tailscale.com/install.sh | sh

In [ ]:
# @title Start Tailscale

from google.colab import userdata
import os
import time

tailscale_key = userdata.get("TAILSCALE_AUTHKEY").strip()

# Store Tailscale state on Google Drive so it survives Colab restarts.
STATE_DIR = "/content/drive/MyDrive/tailscale"
SOCKET = "/var/run/tailscale/tailscaled.sock"

os.makedirs(STATE_DIR, exist_ok=True)
os.makedirs("/var/run/tailscale", exist_ok=True)

# Start Tailscale with persistent state.
!sudo tailscaled \
    --tun=userspace-networking \
    --statedir="$STATE_DIR" \
    --socket="$SOCKET" \
    > /tmp/tailscaled.log 2>&1 &

time.sleep(5)

# Connect using the saved node identity.
!sudo tailscale \
    --socket="$SOCKET" \
    up \
    --authkey="$tailscale_key" \
    --hostname=colab-ollama

print("\n🚀 Tailscale is active!")
print("Tailscale IP:")
!sudo tailscale --socket="$SOCKET" ip -4

In [ ]:
# @title If changing models, cancel the tunnel and execute this to kill Ollama
!pkill ollama